# faststtc — Quick Start

This notebook walks you through the main features of the `faststtc` package using synthetic data. No external data files are required — everything is generated with `generate_spike_trains`. By the end you will know how to:

1. Generate or load spike train data and convert it to a spike matrix
2. Compute pairwise STTC for all units
3. Test whether correlations are statistically significant (z-scores)

All random seeds are fixed to 42 so plots are reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import faststtc as ft

print(f"faststtc version: {ft.__version__}")

## Step 1: Generate or load your data

`faststtc` expects spike times as a **list of arrays** — one array per unit, each containing spike times in **milliseconds**. Here we generate synthetic data using `generate_spike_trains`.

We create two groups:
- **Group A** (units 0–9): correlated — 50% of spikes are shared, so these units tend to fire together
- **Group B** (units 10–19): independent — each unit fires a completely independent Poisson process

We then combine them into a single 20-unit recording of 5 minutes (300,000 ms).

In [ ]:
T_ms = 300_000   # 5-minute recording
rate = 5.0       # 5 Hz mean firing rate

# Group A: correlated (50% shared spikes)
trains_A = ft.generate_spike_trains(N=10, T_ms=T_ms, rate_hz=rate,
                                     correlation=0.5, seed=42)

# Group B: independent
trains_B = ft.generate_spike_trains(N=10, T_ms=T_ms, rate_hz=rate,
                                     correlation=0.0, seed=43)

# Combine into a single recording
all_trains = trains_A + trains_B

# Convert to binary spike matrix (1 ms bins)
spike_matrix = ft.bin_spike_times(all_trains, T_ms=T_ms, bin_size_ms=1)
print(f"Spike matrix shape: {spike_matrix.shape}  (units × time bins)")

# Check firing rates
rates_hz = spike_matrix.sum(axis=1) / (T_ms / 1000)
print(f"Mean firing rate — Group A: {rates_hz[:10].mean():.2f} Hz")
print(f"Mean firing rate — Group B: {rates_hz[10:].mean():.2f} Hz")

## Step 2: Compute pairwise STTC

We pass the spike matrix to `sttc()` with a half-window of Δt = 50 ms. The result is a 20×20 matrix where entry [i, j] is the STTC between unit i and unit j.

The diagonal is NaN (a unit is not compared with itself). Values near 0 indicate independent firing; values near 1 indicate strong co-firing.

In [ ]:
S = ft.sttc(spike_matrix, dt_values=50)
print(f"STTC matrix shape: {S.shape}")
print(f"Group A mean STTC (off-diagonal): {np.nanmean(S[:10, :10]):.3f}")
print(f"Group B mean STTC (off-diagonal): {np.nanmean(S[10:, 10:]):.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(S, vmin=-0.1, vmax=0.6, cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, label='STTC')

# Highlight Group A block
rect = patches.Rectangle((-.5, -.5), 10, 10,
                           linewidth=2, edgecolor='red',
                           facecolor='none', linestyle='--')
ax.add_patch(rect)

ax.set_xlabel('Unit index')
ax.set_ylabel('Unit index')
ax.set_title('Pairwise STTC (Δt = 50 ms)\nRed box = correlated group A')
plt.tight_layout()
plt.show()

## Step 3: Statistical testing

Raw STTC values can be misleading if you don't know what to expect by chance. `zscore_sttc()` builds a null distribution by circularly shifting the spike trains (200 surrogates by default) and returns a z-score for each pair.

A z-score above ~3 is conventionally considered significant.

In [ ]:
z = ft.zscore_sttc(spike_matrix, dt=50, n_shifts=200, seed=42)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(z, vmin=-5, vmax=15, cmap='RdBu_r', aspect='auto')
cbar = plt.colorbar(im, ax=ax, label='z-score')
cbar.ax.axhline(y=3, color='black', linewidth=1.5, linestyle='--')

ax.set_xlabel('Unit index')
ax.set_ylabel('Unit index')
ax.set_title('Z-scored STTC (Δt = 50 ms)\n|z| > 3 = significant')
plt.tight_layout()
plt.show()

# Count significant pairs
triu = np.triu_indices(20, k=1)
z_vals = z[triu]

def count_sig(mask_i, mask_j, threshold=3):
    """Count pairs where both i and j are in the given index masks."""
    count = 0
    for idx in range(len(triu[0])):
        i, j = triu[0][idx], triu[1][idx]
        if mask_i[i] and mask_j[j] and abs(z[i, j]) > threshold:
            count += 1
        elif mask_i[j] and mask_j[i] and abs(z[i, j]) > threshold:
            count += 1
    return count

in_A = np.arange(20) < 10
in_B = np.arange(20) >= 10

n_AA = np.sum(
    (np.abs(z[:10, :10]) > 3) & ~np.eye(10, dtype=bool)
) // 2
n_BB = np.sum(
    (np.abs(z[10:, 10:]) > 3) & ~np.eye(10, dtype=bool)
) // 2
n_AB = np.sum(np.abs(z[:10, 10:]) > 3)

print(f"Significant pairs (|z| > 3):")
print(f"  Within group A:  {n_AA}")
print(f"  Within group B:  {n_BB}")
print(f"  Cross A–B:       {n_AB}")

## Further reading

See [`docs/api.md`](../docs/api.md) for the full API reference, including parameter descriptions, return value shapes, and usage notes for all functions.

The full package documentation is also available at the [GitHub repository](https://github.com/mchini/fastSTTC).